# Perform feature selection on normalized data

## Import libraries

In [1]:
import argparse
import gc
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import psutil
from pycytominer import feature_select
from pycytominer.cyto_utils import output

try:
    cfg = get_ipython().config
    in_notebook = True
except NameError:
    in_notebook = False

In [ ]:
if not in_notebook:
    print("Running as script")
    argparser = argparse.ArgumentParser()
    argparser.add_argument("--num_of_features", type=int, default=1000)
    argparser.add_argument("--num_of_cells_per_well", type=int, default=100)
    argparser.add_argument("--num_of_wells", type=int, default=50)
    args = argparser.parse_args()

    num_of_features = args.num_of_features
    num_of_cells_per_well = args.num_of_cells_per_well
    num_of_wells = args.num_of_wells
else:
    num_of_features = 300
    num_of_cells_per_well = 100
    num_of_wells = 308
    df_shape = (num_of_wells * num_of_cells_per_well, num_of_features)

In [3]:
prior_process = psutil.Process()
prior_mem_info = prior_process.memory_info()

## Perform feature selection

In [4]:
# define operations to be performed on the data
# list of operations for feature select function to use on input profile
feature_select_ops = [
    "variance_threshold",
    "blocklist",
    "drop_na_columns",
    "correlation_threshold",
]

In [5]:
# generate a profile for input data for fs


def generate_toy_data(
    num_of_features: int = 1000,
    num_of_cells_per_well: int = 100,
    num_of_wells: int = 308,
    seed: int = 0,
):
    np.random.seed(seed)
    num_of_rows_total = num_of_wells * num_of_cells_per_well
    output_dict = {
        "Metadata_Well": [],
    }
    for well in range(num_of_wells):
        output_dict["Metadata_Well"].extend([well] * num_of_cells_per_well)

    for feature in range(num_of_features):
        feature_name = f"feature_{feature}"
        output_dict[feature_name] = np.random.normal(0, 1, num_of_rows_total)
    df = pd.DataFrame(output_dict)
    return df


df = generate_toy_data(
    num_of_features=num_of_features,
    num_of_cells_per_well=num_of_cells_per_well,
    num_of_wells=num_of_wells,
)
df.head()

,Metadata_Well,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_290,feature_291,feature_292,feature_293,feature_294,feature_295,feature_296,feature_297,feature_298,feature_299
0,0,1.764052,-0.692224,-0.951221,0.235459,0.454459,2.098793,-1.302832,-1.153722,0.665182,...,-0.681866,0.777416,-0.150975,1.296094,1.615250,0.511771,-0.335374,1.690729,-0.903206,-0.513849
1,0,0.400157,0.450095,-0.052035,-1.699784,1.034151,-0.270473,0.780551,0.623042,-0.171392,...,-0.867100,0.849849,0.274291,1.107074,-0.638470,1.564903,0.191726,1.177723,0.150112,-0.832734
2,0,0.978738,0.824762,0.649247,0.820166,-1.653433,0.650439,0.519204,-0.019524,0.124087,...,0.175421,-0.327349,1.571529,-2.230758,-0.608931,-0.266987,0.771022,0.945128,2.518003,-0.161979
3,0,2.240893,-1.544051,-0.402303,-0.256753,-0.585172,0.329345,-0.744154,1.246493,-0.791892,...,-0.829739,2.305572,1.416862,-0.330355,-0.911360,0.415448,-0.503703,0.819994,0.174548,0.836937
4,0,1.867558,1.058295,-1.493931,-0.450687,-2.595726,-1.593093,-0.616763,-0.563950,-1.313120,...,-0.543686,-2.323647,0.968845,-1.372377,0.576933,-0.131276,2.143337,1.609088,-1.097673,0.928238


In [6]:
metadata_cols = ["Metadata_Well"]
feature_cols = [x for x in df.columns if x not in metadata_cols]
start_time = time.time()
feature_select_df = feature_select(
    df,
    operation=feature_select_ops,
    features=feature_cols,
)
elapsed_time = time.time() - start_time
num_of_features_retained = feature_select_df.shape[1]
percent_of_features_retained = num_of_features_retained / df.shape[1] * 100
print(f"Initial shape: {df.shape}, Final shape: {feature_select_df.shape}")
print(f"Number of features retained: {num_of_features_retained}")
print(f"Percent of features retained: {percent_of_features_retained:.2f}%")
del df
del feature_select_df

Initial shape: (30800, 301), Final shape: (30800, 301)
Number of features retained: 301
Percent of features retained: 100.00%


In [7]:
post_process = psutil.Process()
post_mem_info = post_process.memory_info()

print(f"RSS: {(post_mem_info.rss - prior_mem_info.rss) / (1024 ** 2):.2f} MB")
print(f"VMS: {(post_mem_info.vms - prior_mem_info.vms) / (1024 ** 2):.2f} MB")
print(f"Shared: {(post_mem_info.shared - prior_mem_info.shared) / (1024 ** 2):.2f} MB")
print(f"Text: {(post_mem_info.text - prior_mem_info.text) / (1024 ** 2):.2f} MB")
print(f"Lib: {(post_mem_info.lib - prior_mem_info.lib) / (1024 ** 2):.2f} MB")
print(f"Data: {(post_mem_info.data - prior_mem_info.data) / (1024 ** 2):.2f} MB")
print(f"Dirty: {(post_mem_info.dirty - prior_mem_info.dirty) / (1024 ** 2):.2f} MB")
print(f"Total: {(post_mem_info.rss - prior_mem_info.rss) / (1024 ** 2):.2f} MB")

output_dict = {
    "num_of_features": num_of_features,
    "num_of_cells_per_well": num_of_cells_per_well,
    "num_of_wells": num_of_wells,
    "num_of_features_retained": num_of_features_retained,
    "percent_of_features_retained": percent_of_features_retained,
    "rss_MB": (post_mem_info.rss - prior_mem_info.rss) / (1024**2),
    "vms_MB": (post_mem_info.vms - prior_mem_info.vms) / (1024**2),
    "shared_MB": (post_mem_info.shared - prior_mem_info.shared) / (1024**2),
    "text_MB": (post_mem_info.text - prior_mem_info.text) / (1024**2),
    "lib_MB": (post_mem_info.lib - prior_mem_info.lib) / (1024**2),
    "data_MB": (post_mem_info.data - prior_mem_info.data) / (1024**2),
    "dirty_MB": (post_mem_info.dirty - prior_mem_info.dirty) / (1024**2),
    "total_MB": (post_mem_info.rss - prior_mem_info.rss) / (1024**2),
    "elapsed_time": elapsed_time,
}
output_df = pd.DataFrame(output_dict, index=[0])
output_df

RSS: 154.15 MB
VMS: 184.29 MB
Shared: 2.44 MB
Text: 0.00 MB
Lib: 0.00 MB
Data: 183.11 MB
Dirty: 0.00 MB
Total: 154.15 MB


,num_of_features,num_of_cells_per_well,num_of_wells,num_of_features_retained,percent_of_features_retained,rss_MB,vms_MB,shared_MB,text_MB,lib_MB,data_MB,dirty_MB,total_MB,elapsed_time
0,300,100,308,301,100.0,154.152344,184.289062,2.4375,0.0,0.0,183.109375,0.0,154.152344,0.767771


In [8]:
# write the output to a file
output_file_path = pathlib.Path(
    f"../results_of_memory_profiling/{num_of_features}_features_{num_of_cells_per_well}_cells_per_well_{num_of_wells}_wells.parquet"
).resolve()
output_file_path.parent.mkdir(exist_ok=True, parents=True)
output_df.to_parquet(output_file_path)